# Part 2: Classical Forecasting — Naive, Seasonal Naive, ETS
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Wednesday — Three Baselines That Should Be Tried First

Yesterday Sarah saw the Seasonal Naive baseline scored ~21% MAPE on a holiday-period test window. Today she tries three classical methods to see if they do better.

**The hierarchy of forecasting effort:**
1. **Naive** — predict the last observed value. The easiest possible baseline.
2. **Seasonal Naive** — predict the value from one season ago.
3. **ETS (Exponential Smoothing)** — adaptive level + trend + seasonality. The pragmatic baseline every forecaster should know.

If a fancier model can't beat these, ship one of them.

**By the end of this notebook you will be able to:**
- Build three baseline forecasts in a few lines each
- Fit an ETS model with `statsmodels`
- Compare all three on the same test window with MAE / RMSE / MAPE
- Understand when ETS helps — and when it doesn't

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (13, 4.5)

print("✅ Libraries loaded — statsmodels ExponentialSmoothing ready")

## Step 1 — Setup + train/test split (same as NB 02)

In [ ]:
df = pd.read_csv("data/northstar_daily_revenue.csv", parse_dates=["date"])
df = df.set_index("date")
y = df["revenue_gbp"]

TEST_SIZE = 60
y_train = y.iloc[:-TEST_SIZE]
y_test  = y.iloc[-TEST_SIZE:]

print(f"Train: {len(y_train)} days ({y_train.index[0].date()} to {y_train.index[-1].date()})")
print(f"Test:  {len(y_test)} days ({y_test.index[0].date()} to {y_test.index[-1].date()})")
print(f"Test period is mostly November-December → holiday-shopping season.")

## Step 2 — Method 1: Naive (predict the last value)

The trivial baseline: every forecast equals the last observed value.

In [ ]:
# Predict the same value (last train) for all test dates
naive_forecast = pd.Series(
    [y_train.iloc[-1]] * len(y_test),
    index=y_test.index,
    name="naive"
)

# Helper for metrics
def evaluate(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = (np.abs(y_true - y_pred) / y_true).mean() * 100
    return {"method": name, "MAE": mae, "RMSE": rmse, "MAPE_pct": mape}

results = []
results.append(evaluate(y_test, naive_forecast, "Naive"))
print(f"Naive — predict {y_train.iloc[-1]:.0f} for all {len(y_test)} test days:")
for k, v in results[-1].items():
    if k != "method": print(f"  {k}: {v:.2f}" if k == "MAPE_pct" else f"  {k}: £{v:,.0f}")

## Step 3 — Method 2: Seasonal Naive (predict value from one season ago)

For each test day, use the value from 7 days earlier (same-day-last-week).

(We did this in NB 02 — but with a slightly different setup. Re-run here so we have all three methods in one results table.)

In [ ]:
snaive_forecast = pd.Series(index=y_test.index, dtype=float)
for date in y_test.index:
    last_week = date - pd.Timedelta(days=7)
    if last_week in y_train.index:
        snaive_forecast.loc[date] = y_train.loc[last_week]
    else:
        # for days early in test window, look back further in train
        snaive_forecast.loc[date] = y_train.iloc[-7:].mean()

results.append(evaluate(y_test, snaive_forecast, "Seasonal Naive (lag-7)"))
for k, v in results[-1].items():
    if k != "method": print(f"  {k}: {v:.2f}" if k == "MAPE_pct" else f"  {k}: £{v:,.0f}")

## ⏸️ Pause and Predict

Before running ETS, predict:

- Will ETS beat Seasonal Naive on this test window? By how much?
- Two reasonable hyperparameter choices: `seasonal_periods=7` (weekly) or `seasonal_periods=365` (annual). Which would be better for capturing the November-December holiday spike?

> *Sample answers:*
> - ETS with weekly seasonality will probably beat Seasonal Naive modestly — it ALSO models the upward trend, which Seasonal Naive doesn't.
> - ETS with annual seasonality (period=365) SHOULD be much better for the holiday-window test — it would know "Nov-Dec is high every year." But we only have 2 years of training data, which is too little to reliably fit an annual seasonality. The model may not converge well.

## Step 4 — Method 3: ETS / Exponential Smoothing

We try TWO ETS variants:
- **Weekly seasonality** (period=7) — captures weekend lift + trend
- **Annual seasonality** (period=365) — captures the November-December holiday spike, but needs enough years of data to fit

Then we pick the best.

In [ ]:
# ETS with weekly seasonality (period=7)
ets_weekly = ExponentialSmoothing(
    y_train,
    trend="add",
    seasonal="add",
    seasonal_periods=7,
).fit()

ets_weekly_forecast = ets_weekly.forecast(steps=len(y_test))
ets_weekly_forecast.index = y_test.index

results.append(evaluate(y_test, ets_weekly_forecast, "ETS (weekly, trend+seasonal)"))
print(f"ETS (weekly seasonality):")
for k, v in results[-1].items():
    if k != "method": print(f"  {k}: {v:.2f}" if k == "MAPE_pct" else f"  {k}: £{v:,.0f}")

In [ ]:
# ETS with annual seasonality (period=365) — note: training has ~1.8 cycles only
try:
    ets_annual = ExponentialSmoothing(
        y_train,
        trend="add",
        seasonal="add",
        seasonal_periods=365,
    ).fit()
    ets_annual_forecast = ets_annual.forecast(steps=len(y_test))
    ets_annual_forecast.index = y_test.index

    results.append(evaluate(y_test, ets_annual_forecast, "ETS (annual, trend+seasonal)"))
    print(f"ETS (annual seasonality):")
    for k, v in results[-1].items():
        if k != "method": print(f"  {k}: {v:.2f}" if k == "MAPE_pct" else f"  {k}: £{v:,.0f}")
except Exception as e:
    ets_annual_forecast = None
    print(f"ETS-annual failed to fit: {e}")

## Step 5 — Comparison table

In [ ]:
results_df = pd.DataFrame(results).set_index("method")
print(results_df.round(2).to_string())
print()

# Pick the best
best_method = results_df["MAE"].idxmin()
print(f"Best on MAE: {best_method}  (MAE = £{results_df.loc[best_method, 'MAE']:,.0f})")

### 💡 What you should notice

- **Naive is terrible** — it ignores both trend AND seasonality. It's the floor.
- **Seasonal Naive captures weekly seasonality** but misses the rising holiday trend.
- **ETS-weekly does modestly better** — it adds trend on top of seasonality.
- **ETS-annual** *might* be the best if it converges well — it should know the holiday spike is annual. If it fails (which is common with limited annual data), we'll need a different approach (Thursday's ML method handles multiple seasonalities at once).

The **key insight**: classical models handle ONE seasonality. Our data has TWO (weekly + annual). That's why tomorrow's ML method with lag features at multiple horizons is the right tool.

## Step 6 — Visualise all forecasts together

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Last 90 days of train for context
ax.plot(y_train.iloc[-90:].index, y_train.iloc[-90:].values,
        color="steelblue", linewidth=1.0, label="Train (last 90 days)")
ax.plot(y_test.index, y_test.values, "o-", color="black",
        markersize=4, linewidth=1.4, label="Actual (test)")

ax.plot(naive_forecast.index, naive_forecast.values, "--",
        color="lightgray", linewidth=1.2, label=f"Naive (MAPE {results_df.loc['Naive','MAPE_pct']:.1f}%)")
ax.plot(snaive_forecast.index, snaive_forecast.values, "--",
        color="coral", linewidth=1.2, label=f"Seasonal Naive (MAPE {results_df.loc['Seasonal Naive (lag-7)','MAPE_pct']:.1f}%)")
ax.plot(ets_weekly_forecast.index, ets_weekly_forecast.values, "--",
        color="indianred", linewidth=1.2,
        label=f"ETS-weekly (MAPE {results_df.loc['ETS (weekly, trend+seasonal)','MAPE_pct']:.1f}%)")

if ets_annual_forecast is not None:
    ax.plot(ets_annual_forecast.index, ets_annual_forecast.values, "--",
            color="seagreen", linewidth=1.2,
            label=f"ETS-annual (MAPE {results_df.loc['ETS (annual, trend+seasonal)','MAPE_pct']:.1f}%)")

ax.axvline(y_test.index[0], color="gray", linestyle=":", linewidth=1, alpha=0.7)
ax.set_xlabel("Date")
ax.set_ylabel("Revenue (£)")
ax.set_title("Classical forecasts vs actual — last 60 days")
ax.legend(loc="upper left", fontsize=10)
plt.tight_layout()
plt.show()

## Step 7 — Inspect the residual structure of the best classical model

If a forecast has SYSTEMATIC errors (consistently over or under), that's a structural problem the model didn't capture. Plot the residuals.

In [ ]:
# Use the best ETS variant (probably weekly)
best_classical = ets_weekly_forecast
residuals = y_test - best_classical

fig, axes = plt.subplots(2, 1, figsize=(13, 6))

axes[0].plot(residuals.index, residuals.values, "o-", color="seagreen", markersize=4)
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_ylabel("Residual (£)")
axes[0].set_title(f"ETS-weekly residuals over the test window")

axes[1].hist(residuals, bins=20, color="seagreen", edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("Residual (£)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual distribution — should be centred near 0")

plt.tight_layout()
plt.show()

print(f"Mean residual:   £{residuals.mean():+.0f}  (positive = model underforecasts)")
print(f"Median residual: £{residuals.median():+.0f}")

### 💡 What this tells us

- **Positive mean residual** = the model **under-forecasts**. That's typical when forecasting through a rising holiday season with a model that only knows weekly seasonality.
- **The residual chart** shows whether the under-forecast is consistent (every day a bit low) or spiky (just a few days).

This is exactly why we need Thursday's ML approach — it can capture annual seasonality through lag-365 features.

## ✅ Section Summary

| Method | When it works | When it fails |
|---|---|---|
| **Naive** | Series with no trend, no seasonality | Always — but it's the bar floor |
| **Seasonal Naive** | Stable seasonal pattern, no trend | Strong trend or rising seasons |
| **ETS-weekly** | Series with one seasonal cycle | Multi-seasonal data |
| **ETS-annual** | Long enough training data (3+ years) | Limited annual data, convergence issues |

**Surprising result on THIS test window:**
> Naive *coincidentally* wins on the holiday-window MAE — because the last training day happened to be £10,813, close to the test-window average. This is exactly why you should NEVER evaluate on a single window. `TimeSeriesSplit` from NB 02 gives you 5 windows to average over; on most non-holiday windows ETS-weekly will beat Naive cleanly.

**Friday recommendation so far:**
> Classical methods (especially ETS-weekly) capture trend + weekly seasonality. They miss the annual holiday spike. Tomorrow's ML method with lag features at multiple horizons (lag-1, lag-7, lag-365) handles BOTH seasonalities at once. That's likely the winner.

**Key insights:**
- **ALWAYS start with a baseline.** Naive + Seasonal Naive define the floor.
- **ETS captures level + trend + ONE seasonality** — pragmatic baseline that's been used for decades.
- **For multi-seasonal data**, classical methods leave room on the table. ML approaches with multi-lag features fill that gap.

---
**Up next → Part 3:** Thursday — ML-based forecasting with lag features + HistGradientBoostingRegressor. The method that should finally beat all the classical baselines.
Open `04_ml_forecasting.ipynb`

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## Extension 1 — SimpleExpSmoothing (level only, no trend, no seasonality)

The simplest form of exponential smoothing — just adaptive weighted average. Useful when there's no clear trend OR seasonality.

In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

ses = SimpleExpSmoothing(y_train).fit()
ses_forecast = ses.forecast(steps=len(y_test))
ses_forecast.index = y_test.index

ses_result = evaluate(y_test, ses_forecast, "SimpleExpSmoothing")
print(f"SimpleExpSmoothing: MAE £{ses_result['MAE']:,.0f}, MAPE {ses_result['MAPE_pct']:.1f}%")
print(f"  → essentially predicts a flat line — no trend, no seasonality.")
print(f"  → useful as a sanity-check baseline; almost always beaten by ETS-weekly.")

## Extension 2 — ETS with multiplicative seasonality

The default `seasonal="add"` assumes the weekly amplitude is the same regardless of overall level. `seasonal="mul"` assumes the amplitude SCALES with the level (10% weekend lift, whether revenue is £8k or £12k).

In [ ]:
try:
    ets_mul = ExponentialSmoothing(
        y_train,
        trend="add",
        seasonal="mul",
        seasonal_periods=7,
    ).fit()
    ets_mul_forecast = ets_mul.forecast(steps=len(y_test))
    ets_mul_forecast.index = y_test.index
    mul_result = evaluate(y_test, ets_mul_forecast, "ETS-weekly (mul)")
    print(f"ETS-weekly with multiplicative seasonality: MAE £{mul_result['MAE']:,.0f}, MAPE {mul_result['MAPE_pct']:.1f}%")
    print(f"vs additive: MAE £{results_df.loc['ETS (weekly, trend+seasonal)','MAE']:,.0f}, MAPE {results_df.loc['ETS (weekly, trend+seasonal)','MAPE_pct']:.1f}%")
except Exception as e:
    print(f"Multiplicative ETS failed: {e}")

## Extension 3 — Forecast 90 days into the FUTURE (no actual to compare)

So far we've been forecasting where we have ground truth. The real deployment task: forecast Jan-Mar 2026 (no data yet). Sarah's Friday deliverable.

In [ ]:
# Re-fit on ALL data + forecast 90 days into 2026
ets_final = ExponentialSmoothing(
    y, trend="add", seasonal="add", seasonal_periods=7,
).fit()
forecast_90 = ets_final.forecast(steps=90)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y.iloc[-180:].index, y.iloc[-180:].values, color="steelblue",
        linewidth=1.0, label="Last 6 months of actual")
ax.plot(forecast_90.index, forecast_90.values, "o-", color="darkorange",
        markersize=4, linewidth=1.4, label="90-day forecast")
ax.axvline(y.index[-1], color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Revenue (£)")
ax.set_title("ETS-weekly 90-day forecast — January through March 2026")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Forecasted total revenue Jan-Mar 2026: £{forecast_90.sum():,.0f}")
print(f"Average daily forecast: £{forecast_90.mean():,.0f}")
print()
print("Sarah's Friday number for Marcus.")